# Mini fine-tuning lab: LoRA on a model small enough to run yourself

The main lesson replays eleven fine-tuning runs of a **7B** model. Those needed a
data-centre GPU. This notebook runs the same experiment at a size you can afford:

- **Model:** `Qwen/Qwen2.5-0.5B-Instruct` (0.49B parameters, same family as the lesson's model)
- **Task:** WikiSQL, English question → SQL query, same prompt format as the lesson
- **Metric:** execution accuracy: run the generated SQL and check that it returns the right rows

**Hardware.** A free Google Colab T4 GPU, or an Apple-silicon Mac. With the default
settings, training takes roughly 10–20 minutes on a T4.

**What you will do**
1. Score the model *before* fine-tuning.
2. Fine-tune it with LoRA (or full fine-tuning, or QLoRA on a CUDA GPU).
3. Score it again, and use a **paired test** to check whether the improvement is real.
4. Change one setting at a time and see what moves.

Your numbers will not match the lesson's. This is a smaller model, a smaller
training set and fewer steps, so it is a different experiment. That is part of the point.

In [ ]:
# Colab: install the libraries. On your own machine, run this once in a terminal instead.
# !pip install -q "transformers>=4.45" "peft>=0.13" "datasets>=2.20" accelerate bitsandbytes

## 1. Settings

Change **one thing at a time**. Otherwise you cannot tell which change caused the result.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
METHOD = "lora"        # "lora", "full", or "qlora" (qlora needs a CUDA GPU)
RANK = 8               # LoRA rank r
ALPHA = 2 * RANK       # the lesson uses alpha = 2r at every rank
LR = 2e-4 if METHOD != "full" else 1e-5   # same rates as the lesson
N_TRAIN = 2000         # training examples (the lesson used about 56,000)
N_EVAL = 200           # held-out questions to score (the lesson used 1,000)
EPOCHS = 1
BATCH = 8
MAX_LEN = 256          # tokens per example; WikiSQL prompts are short
SEED = 42

In [ ]:
import math, random, sqlite3, time
import torch

random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else (
    torch.float16 if DEVICE == "cuda" else torch.float32)
if METHOD == "qlora" and DEVICE != "cuda":
    raise SystemExit("QLoRA uses bitsandbytes, which needs a CUDA GPU. Use METHOD = 'lora'.")
print("device:", DEVICE, "| dtype:", DTYPE)

## 2. The data

WikiSQL stores each query as a small structure (which column, which aggregate, which
conditions). We turn it into SQL text. Columns are called `col0`, `col1`, … and the
table is always `"table"`. The model has never seen that convention, so it has to
learn it from fine-tuning.

In [ ]:
from datasets import load_dataset

# The parquet conversion loads in every version of `datasets`.
ds = load_dataset("Salesforce/wikisql", revision="refs/convert/parquet")

AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<", "OP"]

def render_sql(sql):
    sel, agg = f"col{sql['sel']}", AGG_OPS[sql["agg"]]
    out = f'SELECT {f"{agg}({sel})" if agg else sel} FROM "table"'
    c = sql["conds"]
    triples = list(zip(c["column_index"], c["operator_index"], c["condition"]))
    if triples:
        out += " WHERE " + " AND ".join(
            f"col{ci} {COND_OPS[o]} '" + str(v).strip().replace("'", "''") + "'"
            for ci, o, v in triples)
    return out

SYSTEM = ('You translate questions into SQL. Answer with one SQL query and '
          'nothing else. Refer to columns as col0, col1, ... and the table '
          'as "table".')

def build_prompt(question, headers, types):
    schema = "\n".join(f"  col{i}: {h} ({t})" for i, (h, t) in enumerate(zip(headers, types)))
    return f'Table "table":\n{schema}\n\nQuestion: {question.strip()}\n\nSQL:'

train_rows = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
eval_rows = ds["validation"].select(range(N_EVAL))

ex = train_rows[0]
print(build_prompt(ex["question"], ex["table"]["header"], ex["table"]["types"]))
print("→", render_sql(ex["sql"]))

## 3. Scoring by execution

Two queries can be written differently and still return the same rows. So we build
each table in an in-memory SQLite database, run both the generated query and the
correct one, and compare the rows.

In [ ]:
def execute(table, query):
    try:
        types = table["types"] or ["text"] * len(table["header"])
        conn = sqlite3.connect(":memory:")
        cols = ", ".join(f'col{i} {"REAL" if t == "real" else "TEXT"}' for i, t in enumerate(types))
        conn.execute(f'CREATE TABLE "table" ({cols})')
        rows = []
        for row in table["rows"]:
            vals = []
            for v, t in zip(row, types):
                if t == "real":
                    try: vals.append(float(str(v).replace(",", "")))
                    except (ValueError, TypeError): vals.append(None)
                else:
                    vals.append(str(v))
            rows.append(vals)
        conn.executemany(f'INSERT INTO "table" VALUES ({",".join("?" * len(types))})', rows)
        out = conn.execute(query).fetchall()
        conn.close()
        return out, None
    except Exception as e:
        return None, str(e)

def execution_correct(table, predicted, gold):
    p, err = execute(table, predicted)
    g, _ = execute(table, gold)
    return err is None and p == g

## 4. Load the model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

kwargs = {"dtype": DTYPE}
if METHOD == "qlora":
    from transformers import BitsAndBytesConfig
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    kwargs["device_map"] = {"": 0}
model = AutoModelForCausalLM.from_pretrained(MODEL, **kwargs)
if METHOD != "qlora":
    model.to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 5. Score the model before fine-tuning

This is the baseline. Most of its mistakes are not bad reasoning. It just does not
know the `col0` / `"table"` convention.

In [ ]:
@torch.no_grad()
def generate_all(model, rows, batch=16, max_new_tokens=64):
    model.eval()
    preds = []
    for i in range(0, len(rows), batch):
        chunk = rows.select(range(i, min(i + batch, len(rows))))
        texts = [tok.apply_chat_template(
                    [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": build_prompt(q, t["header"], t["types"])}],
                    tokenize=False, add_generation_prompt=True)
                 for q, t in zip(chunk["question"], chunk["table"])]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for seq in out[:, enc["input_ids"].shape[1]:]:
            preds.append(tok.decode(seq, skip_special_tokens=True).strip().split("\n")[0].strip("` "))
    return preds

def score(preds, rows):
    return [int(execution_correct(r["table"], p, render_sql(r["sql"]))) for p, r in zip(preds, rows)]

t0 = time.time()
before_preds = generate_all(model, eval_rows)
before = score(before_preds, eval_rows)
print(f"before fine-tuning: {sum(before)}/{len(before)} = {sum(before)/len(before):.1%} "
      f"execution accuracy ({time.time()-t0:.0f}s)")
for p, r in list(zip(before_preds, eval_rows))[:3]:
    print("  model:", p, "\n  gold: ", render_sql(r["sql"]))

## 6. Attach LoRA (or don't)

With LoRA, the original weights are frozen and only the small `A` and `B` matrices
train. The adapted modules are the four attention projections, as in the lesson.

In [ ]:
if METHOD in ("lora", "qlora"):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if METHOD == "qlora":
        model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=RANK, lora_alpha=ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    model.print_trainable_parameters()
else:
    print(f"full fine-tuning: all {sum(p.numel() for p in model.parameters()):,} parameters train")

## 7. Build training examples

Each example is the prompt followed by the correct SQL. The loss is computed **only on
the SQL tokens** (the prompt's labels are set to -100), so the model learns to write
the answer, not to repeat the question.

In [ ]:
def encode(row):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": build_prompt(row["question"], row["table"]["header"], row["table"]["types"])}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    answer = render_sql(row["sql"]) + tok.eos_token
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tok(answer, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return ids, labels

train = [encode(r) for r in train_rows]

def batches():
    order = list(range(len(train)))
    random.shuffle(order)
    for i in range(0, len(order), BATCH):
        chunk = [train[j] for j in order[i:i + BATCH]]
        L = max(len(ids) for ids, _ in chunk)
        ids = torch.full((len(chunk), L), tok.pad_token_id)
        lab = torch.full((len(chunk), L), -100)
        att = torch.zeros((len(chunk), L), dtype=torch.long)
        for k, (x, y) in enumerate(chunk):   # right-pad for training
            ids[k, :len(x)] = torch.tensor(x); lab[k, :len(y)] = torch.tensor(y); att[k, :len(x)] = 1
        yield ids.to(model.device), lab.to(model.device), att.to(model.device)

## 8. Train

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
steps = EPOCHS * math.ceil(len(train) / BATCH)
warmup = max(1, int(0.03 * steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) *
                                          0.5 * (1 + math.cos(math.pi * min(1, s / steps))))
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

model.train()
step, t0, losses = 0, time.time(), []
for epoch in range(EPOCHS):
    for ids, lab, att in batches():
        loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        losses.append(loss.item()); step += 1
        if step % 25 == 0 or step == steps:
            print(f"step {step}/{steps}  loss {sum(losses[-25:])/len(losses[-25:]):.4f}  "
                  f"{time.time()-t0:.0f}s")
if DEVICE == "cuda":
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 9. Score again, and test whether the change is real

Before and after were scored on the **same** questions, so this is a paired
comparison. As in the lesson's statistics section, only the questions where the two
disagree carry information.

In [ ]:
after_preds = generate_all(model, eval_rows)
after = score(after_preds, eval_rows)
print(f"before: {sum(before)/len(before):.1%}   after: {sum(after)/len(after):.1%}")

gained = sum(1 for x, y in zip(before, after) if y and not x)   # wrong before, right after
lost = sum(1 for x, y in zip(before, after) if x and not y)     # right before, wrong after
n = gained + lost
p = min(1.0, 2 * sum(math.comb(n, i) for i in range(min(gained, lost) + 1)) / 2 ** n) if n else 1.0
print(f"questions fixed: {gained}, questions broken: {lost}, McNemar exact p = {p:.4g}")

rng = random.Random(SEED)
d = [y - x for x, y in zip(before, after)]
boot = sorted(sum(d[rng.randrange(len(d))] for _ in d) / len(d) for _ in range(2000))
print(f"paired bootstrap 95% interval for the gain: {boot[50]:+.1%} to {boot[1949]:+.1%}")

## 10. Your turn

Change one setting in section 1, re-run from there, and write down what changes.
Each question has more than one reasonable answer.

1. **Rank.** Try `RANK = 2` and `RANK = 64`. The lesson found no measurable
   accuracy difference across ranks at 7B. Does that hold at 0.5B, where the model
   has less to start from?
2. **Data.** Try `N_TRAIN = 500` and `N_TRAIN = 8000`. Which helps more here: more
   rank, or more data?
3. **Full fine-tuning.** Set `METHOD = "full"`. On a T4, watch the memory. Is the
   accuracy different from LoRA's? Is the difference significant under the paired
   test above?
4. **Seeds.** Run the same setting with `SEED = 1` and `SEED = 2`. How far does
   accuracy move with nothing but the seed changed? Compare that with the gaps the
   lesson reports between methods. What does that tell you about one-seed comparisons?
5. **Learning rate.** The lesson gave full fine-tuning `1e-5`. Try `3e-5` with
   `METHOD = "full"`. If the ranking between methods flips, what does that say about
   claims like "LoRA matches full fine-tuning"?